# Intention CNN — Character-Level Micro-Classifier for "im X" Parsing

**Task**: Classify `im X` utterances into:
- `name` — "I'm Sarah" / "im zara" / "im bob"
- `state` — "I'm bored" / "im tired" / "im hungry"
- `apology` — "im sorry" / "im so sorry"
- `intent` — "im going to NYC" / "im about to sleep"
- `chitchat` — "im just vibing" / "im just saying"
- `other` — anything else

**Architecture**: ~5M params, 3-layer 1D-CNN over character trigrams
**Training**: Synthetic data, 50K examples, 5 epochs
**Goal**: Replace Star's naive `im (.*)` regex with a learned classifier

**Key insight**: Character-level convolution inherently distinguishes:
- `"I'm Sarah"` (capital after `'` = name)
- `"I'm bored"` (lowercase after `'` = adjective = state)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Imports & Config
# ─────────────────────────────────────────────────────────────
import os
import random
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ─────────────────────────────────────────────────────────────
# Label definitions
# ─────────────────────────────────────────────────────────────
LABEL2ID = {
    "name":       0,
    "state":      1,
    "apology":    2,
    "intent":     3,
    "chitchat":   4,
    "other":      5,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = len(LABEL2ID)
print(f"Classes: {NUM_CLASSES}", LABEL2ID)

## 1. Synthetic Data Generation

We generate training examples from templates with realistic noise:
- Capitalization variation
- Spacing ("im" vs "i m" vs "i'm")
- Typos and misspellings
- Trailing punctuation

In [ ]:
# ─────────────────────────────────────────────────────────────
# Name lists — real names that follow "im [Name]"
# ─────────────────────────────────────────────────────────────
FIRST_NAMES = [
    "Sarah", "Zach", "Alex", "Jordan", "Taylor", "Morgan",
    "Casey", "Riley", "Quinn", "Avery", "Charlie", "Dakota",
    "Jamie", "Drew", "Skylar", "Parker", "Reese", "Hayden",
    "Peyton", "Finley", "Emerson", "Blair", "Sage", "River",
    # Short names
    "Ann", "Ben", "Liz", "Bob", "Sam", "Jo", "Ray",
    # Foreign / diverse names
    "Priya", "Chen", "Amara", "Kofi", "Yuki", "Sven",
    "Lena", "Raj", "Fatima", "Ivan", "Nia", "Mateo",
]

# Adjectives / states that follow "im [adjective]"
STATE_ADJECTIVES = [
    "bored", "tired", "hungry", "thirsty", "sleepy", "lazy",
    "curious", "happy", "sad", "angry", "nervous", "calm",
    "excited", "worried", "confused", "frustrated", "grateful",
    "alone", "lost", "stuck", "ready", "done", "fine",
    "weird", "weird", "sober", "fried", "zonked",
]

# Apology phrases
APOLOGY_TEMPLATES = [
    "sorry", "so sorry", "really sorry", "super sorry",
    "so sry", "sry", "my bad", "my fault", "apologies",
]

# Intent phrases — starts with "im" + verb phrase
INTENT_STARTS = [
    "going to", "about to", "heading to", "leaving for",
    "starting to", "trying to", "gonna", "finna",
]

# Chitchat — "im just..." / "im just saying"
CHITCHAT_STARTS = [
    "just vibing", "just saying", "just thinking",
    "just messing around", "just joshing", "just kidding",
    "just checking in", "just lurking", "just passing through",
    "just rambling", "just talking", "just being",
]

# Spacing / punctuation variants
IM_VARIANTS = ["im", "i m", "i'm", "I'm", "IM", "i'm"]

# Common typos for "im"
def noisy_im(variant):
    if random.random() < 0.1:
        # Misspell
        typos = {"im": ["ihm", "mi", "im", "im\n"], "i m": ["1m", "l m"]}
        return random.choice(typos.get(variant.lower(), [variant]))
    return variant

def noisy_capitalize(text):
    if random.random() < 0.3:
        return text.upper()
    elif random.random() < 0.3:
        return text.title()
    return text

def noisy_trailing(text):
    suffixes = ["", "", "", ".", "...", "!!", "?"]
    return text + random.choice(suffixes)

def generate_name_example():
    variant = noisy_im(random.choice(IM_VARIANTS))
    name = random.choice(FIRST_NAMES)
    if random.random() < 0.2:
        name = name.lower()  # "im sarah" not "im Sarah"
    text = f"{variant} {name}"
    text = noisy_capitalize(text)
    text = noisy_trailing(text)
    return text.strip()

def generate_state_example():
    variant = noisy_im(random.choice(IM_VARIANTS))
    adj = random.choice(STATE_ADJECTIVES)
    text = f"{variant} {adj}"
    text = noisy_capitalize(text)
    text = noisy_trailing(text)
    return text.strip()

def generate_apology_example():
    variant = noisy_im(random.choice(IM_VARIANTS))
    phrase = random.choice(APOLOGY_TEMPLATES)
    text = f"{variant} {phrase}"
    text = noisy_capitalize(text)
    text = noisy_trailing(text)
    return text.strip()

def generate_intent_example():
    variant = noisy_im(random.choice(IM_VARIANTS))
    start = random.choice(INTENT_STARTS)
    # Random completion
    rest = random.choice(["NYC", "the store", "to bed", "to sleep",
                          "to work", "dinner", "class", "the gym",
                          "practice", "a movie", "home", "out"])
    text = f"{variant} {start} {rest}"
    text = noisy_capitalize(text)
    text = noisy_trailing(text)
    return text.strip()

def generate_chitchat_example():
    variant = noisy_im(random.choice(IM_VARIANTS))
    start = random.choice(CHITCHAT_STARTS)
    text = f"{variant} {start}"
    text = noisy_capitalize(text)
    text = noisy_trailing(text)
    return text.strip()

def generate_other_example():
    # Negative examples: things that look like "im X" but aren't
    others = [
        "I'm\ximpossible to reach",  # word after
        "I'm\tgoing",
        "I'ma go",
        "imma go",
        "imo",
        "imo it's fine",
        "imgonnawork",
        "im 2 cool 4 u",  # slang
        "i meant im gonna",
        "wait im not sure",
        "actually im good",
        "nah im good",
        "lol im dead",
        "im\xa0tired",  # non-breaking space
    ]
    return random.choice(others)

def generate_all(k=8000):
    """Generate k examples per class = 48K total"""
    samples = []
    generators = [
        ("name",     generate_name_example),
        ("state",    generate_state_example),
        ("apology",  generate_apology_example),
        ("intent",   generate_intent_example),
        ("chitchat", generate_chitchat_example),
        ("other",    generate_other_example),
    ]
    for label, gen in generators:
        for _ in range(k):
            samples.append({"text": gen(), "label": label})
    random.shuffle(samples)
    return samples

# Generate dataset
print("Generating synthetic data...")
raw_data = generate_all(k=8000)
print(f"Total samples: {len(raw_data)}")

# Preview
for label in LABEL2ID:
    ex = next(s["text"] for s in raw_data if s["label"] == label)
    print(f"  {label:10s}: \"{ex}\"")

## 2. Character Vocabulary & Encoding

We encode text as character trigram integers — the CNN learns 3-char patterns directly.

In [ ]:
# Character vocabulary — map each char to an integer index
# Include common punctuation, spaces, and a padding token
CHARS = (
    " abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "'\"-.,!?;:\n\t"
    "0123456789"
)
PAD_CHAR = 0
UNK_CHAR = 1

char2idx = {c: i + 2 for i, c in enumerate(CHARS)}
idx2char = {i + 2: c for i, c in enumerate(CHARS)}
idx2char[PAD_CHAR] = "<PAD>"
idx2char[UNK_CHAR] = "<UNK>"
VOCAB_SIZE = len(char2idx) + 2  # + PAD + UNK

MAX_LEN = 48  # max characters per input

def encode(text, max_len=MAX_LEN):
    """Encode string to character integer sequence, padded to max_len."""
    encoded = []
    for ch in text[:max_len]:
        encoded.append(char2idx.get(ch, UNK_CHAR))
    # Pad
    while len(encoded) < max_len:
        encoded.append(PAD_CHAR)
    return encoded

def decode(encoded):
    return "".join(idx2char.get(i, "<") for i in encoded)

# Test
ex = "I'm Sarah"
enc = encode(ex)
print(f"Original: \"{ex}\"")
print(f"Encoded:  {enc}")
print(f"Decoded:  \"{decode(enc)}\"")
print(f"Vocab size: {VOCAB_SIZE}")

## 3. Dataset & DataLoader

In [ ]:
class IntentionDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        x = torch.tensor(encode(text), dtype=torch.long)
        y = torch.tensor(LABEL2ID[label], dtype=torch.long)
        return x, y


# Split 80/10/10
texts = [s["text"] for s in raw_data]
labels = [s["label"] for s in raw_data]

X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

train_ds = IntentionDataset(X_train, y_train)
val_ds   = IntentionDataset(X_val,   y_val)
test_ds  = IntentionDataset(X_test,  y_test)

BATCH_SIZE = 256
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## 4. Model — Character-Level 1D CNN

Architecture:
```
Input (MAX_LEN, VOCAB_SIZE)
  ↓ Embedding(VOCAB_SIZE, 64)
  ↓ 1D-CNN(64→128, kernel=3, ReLU) × 3 with batch norm + dropout
  ↓ GlobalMaxPool
  ↓ FC(128→64, ReLU, dropout=0.3)
  ↓ FC(64→6)
```

Total params: ~180K (tiny!)

In [ ]:
class IntentionCNN(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=64, num_classes=NUM_CLASSES):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Three parallel conv layers with different kernel sizes (trigram, 4-gram, 5-gram)
        self.conv3 = nn.Conv1d(embed_dim, 128, kernel_size=3, padding=1)  # trigram
        self.conv4 = nn.Conv1d(embed_dim, 128, kernel_size=4, padding=2)  # 4-gram
        self.conv5 = nn.Conv1d(embed_dim, 128, kernel_size=5, padding=2)  # 5-gram
        
        self.bn3 = nn.BatchNorm1d(128)
        self.bn4 = nn.BatchNorm1d(128)
        self.bn5 = nn.BatchNorm1d(128)
        
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()
        
        # After cat: 128*3 = 384
        self.fc1 = nn.Linear(128 * 3, 64)
        self.fc2 = nn.Linear(64, num_classes)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        # x: (batch, seq_len)
        x = self.embedding(x)                    # (batch, seq_len, embed_dim)
        x = x.permute(0, 2, 1)                   # (batch, embed_dim, seq_len)
        
        c3 = self.relu(self.bn3(self.conv3(x)))  # (batch, 128, seq_len)
        c4 = self.relu(self.bn4(self.conv4(x)))
        c5 = self.relu(self.bn5(self.conv5(x)))
        
        # Global max pooling over each conv output
        c3 = c3.max(dim=2)[0]  # (batch, 128)
        c4 = c4.max(dim=2)[0]
        c5 = c5.max(dim=2)[0]
        
        x = torch.cat([c3, c4, c5], dim=1)      # (batch, 384)
        x = self.dropout(x)
        x = self.relu(self.fc1(x))              # (batch, 64)
        x = self.dropout(x)
        x = self.fc2(x)                          # (batch, num_classes)
        return x

model = IntentionCNN().to(DEVICE)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable:    {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Quick forward test
x_sample, _ = next(iter(train_dl))
out = model(x_sample.to(DEVICE))
print(f"Output shape: {out.shape}")  # (batch, 6)
print(f"Output logits: {out[:2]}")

## 5. Training

In [ ]:
EPOCHS = 5
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels

best_val_acc = 0
best_model_state = None

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>9} | {'Val Acc':>8} | {'LR':>6}")
print("-" * 65)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_dl, optimizer, criterion, DEVICE)
    val_loss, val_acc, _, _ = eval_epoch(model, val_dl, criterion, DEVICE)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    
    marker = " *" if val_acc > best_val_acc else ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>9.4f} | {val_loss:>9.4f} | {val_acc:>8.4f} | {lr:>6.2e}{marker}")

print(f"\nBest val accuracy: {best_val_acc:.4f}")

## 6. Evaluation — Test Set

In [ ]:
# Load best model
model.load_state_dict(best_model_state)
model.to(DEVICE)

test_loss, test_acc, preds, labels = eval_epoch(model, test_dl, criterion, DEVICE)
print(f"Test accuracy: {test_acc:.4f}\n")

# Classification report
print(classification_report(labels, preds, target_names=list(LABEL2ID.keys())))

# Confusion matrix
cm = confusion_matrix(labels, preds)
print("Confusion matrix:")
print(f"{'':>12}", end="")
for lbl in LABEL2ID:
    print(f"{lbl:>10}", end="")
print()
for i, row in enumerate(cm):
    print(f"{list(LABEL2ID.keys())[i]:>12}", end="")
    for val in row:
        print(f"{val:>10}", end="")
    print()

## 7. Qualitative Examples — Spot Check

In [ ]:
model.eval()

def predict(text):
    enc = torch.tensor(encode(text), dtype=torch.long).unsqueeze(0).to(DEVICE)
    logits = model(enc)
    probs = torch.softmax(logits, dim=1)
    pred = logits.argmax(1).item()
    conf = probs[0, pred].item()
    return ID2LABEL[pred], conf

test_cases = [
    # Name
    "I'm Sarah",
    "im zara",
    "im bob",
    "I'M ALEX",
    # State
    "I'm bored",
    "im tired",
    "im so hungry rn",
    # Apology
    "im sorry",
    "im so sorry about that",
    "Im my bad",
    # Intent
    "im going to NYC",
    "im about to sleep",
    "imma head out",
    # Chitchat
    "im just vibing",
    "im just saying",
    "im just rambling",
    # Other
    "imgonnawork",
    "imo it's fine",
    "actually im good",
    "nah im good",
    # Edge cases
    "I'm\xa0tired",   # non-breaking space
    "i meant im gonna",
    "wait im not sure",
    "lol im dead",
]

print(f"{'Input':<30} {'Prediction':<12} {'Confidence'}")
print("-" * 58)
for text in test_cases:
    pred, conf = predict(text)
    print(f"{text:<30} {pred:<12} {conf:.4f}")

## 8. Export — PyTorch → ONNX

In [ ]:
import os

ONNX_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "intention_cnn.onnx")

# Model must be in eval mode
model.eval()

# Dummy input for ONNX export
dummy = torch.randint(0, VOCAB_SIZE, (1, MAX_LEN), dtype=torch.long)

# Export
torch.onnx.export(
    model,
    dummy,
    ONNX_PATH,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
    opset_version=13,
)
print(f"ONNX model saved to: {ONNX_PATH}")
print(f"File size: {os.path.getsize(ONNX_PATH) / 1024:.1f} KB")

# Verify ONNX model loads correctly
import onnx
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("ONNX model verified OK")

## 9. Save Artifacts

In [ ]:
import json

ARTIFACTS_DIR = os.path.dirname(os.path.abspath(__file__))

# Save model weights
MODEL_PATH = os.path.join(ARTIFACTS_DIR, "intention_cnn.pt")
torch.save({
    "state_dict": best_model_state,
    "vocab_size": VOCAB_SIZE,
    "embed_dim": 64,
    "num_classes": NUM_CLASSES,
    "max_len": MAX_LEN,
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "chars": CHARS,
}, MODEL_PATH)
print(f"Model saved: {MODEL_PATH}")

# Save config
CONFIG_PATH = os.path.join(ARTIFACTS_DIR, "intention_cnn_config.json")
config = {
    "vocab_size": VOCAB_SIZE,
    "embed_dim": 64,
    "num_classes": NUM_CLASSES,
    "max_len": MAX_LEN,
    "label2id": LABEL2ID,
    "chars": CHARS,
    "test_acc": float(test_acc),
    "best_val_acc": float(best_val_acc),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "model_path": MODEL_PATH,
    "onnx_path": ONNX_PATH,
}
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved: {CONFIG_PATH}")
print(f"\nArtifacts ready for integration into Starfire!")

## 10. Integration into Starfire (Rust)

Once the model is exported, add it to Starfire:
```rust
// lib/im_classifier/mod.rs
pub struct ImClassifier {
    model: candle::Module,
    tokenizer: CharTokenizer,
    config: ClassifierConfig,
}

/// Classify an "im X" utterance.
pub fn classify(&self, text: &str) -> ImIntention {
    let encoded = self.tokenizer.encode(text, self.config.max_len);
    let tensor = candle::Tensor::from_slice(&encoded, [1, self.config.max_len])
        .unwrap()
        .to_device(&self.device);
    let logits = self.model.forward(&tensor);
    let pred = logits.argmax(2).squeeze(0).item::<u32>().unwrap();
    ImIntention::from_id(pred)
}
```

Wire into `InputNormalizer` or before the KG reasoning step:
```rust
pub fn classify_im_intention(input: &str) -> ImIntention {
    // "im" prefix detected by normalizer
    // Dispatch based on classifier output:
    match classifier.classify(input) {
        ImIntention::Name => { /* store as user identity */ }
        ImIntention::State => { /* record emotional state */ }
        ImIntention::Apology => { /* emit empathy response */ }
        ImIntention::Intent => { /* update user goals */ }
        ImIntention::Chitchat => { /* pass through to KG */ }
        ImIntention::Other => { /* pass through to KG */ }
    }
}
```